# Iniciando o Spark

In [ ]:
## Bloco de codigo para instalar versao especifica dos pacotes
!pip install pyspark

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Trusted_base_telco") \
    .getOrCreate()

# Importando bibliotecas

In [ ]:
import os
import pytz
import datetime
from datetime import datetime
#from pyspark.sql.types import *
#from pyspark.sql.functions import count, avg
#import sys
#import numpy as np
#from datetime import datetime
#from pyspark.sql import SQLContext
#from datetime import timedelta
#from datetime import date
#from dateutil.relativedelta import relativedelta
#from pyspark.sql.functions import udf, lpad, translate

# Funções auxiliares e variáveis

In [ ]:
# Função de log
def log():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S') + " >>>"

# Timestamp de processamento (com hora/minuto/segundo)
agora = datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc = agora.strftime("%Y%m%d%H%M%S")

# Data da execução (AAAAmmdd)
PROCESS_DATE = datetime.now().strftime("%Y%m%d")

# Período de referência (AAAAmm)
REF_PERIOD = datetime.now().strftime("%Y%m")

#Alterar o path_padrao caso seus arquivos não estejam nesse mesmo caminho
path_padrao = "/content/gdrive/Othercomputers/Meu laptop"

# Buckets e nomes de saída
bucket_base = "base_telco"
bucket_raw = f"{path_padrao}/Database_raw/base_telco"
bucket_trusted = f"{path_padrao}/Database_trusted/base_telco"
bucket_control = f"{path_padrao}/Database_control/base_telco"
output_trusted = f"trusted_{bucket_base}"

# Prints para conferência
print("PROCESS_DATE:", PROCESS_DATE)
print("REF_PERIOD:", REF_PERIOD)
print("dthproc:", dthproc)
print("bucket_trusted:", bucket_trusted)
print("bucket_raw:", bucket_raw)
print("bucket_control:", bucket_control)


PROCESS_DATE: 20260210
REF_PERIOD: 202602
dthproc: 20260210091120
bucket_trusted: /content/gdrive/Othercomputers/Meu laptop/Database_trusted/base_telco
bucket_raw: /content/gdrive/Othercomputers/Meu laptop/Database_raw/base_telco
bucket_control: /content/gdrive/Othercomputers/Meu laptop/Database_control/base_telco


# Leitura dos dados na camada Raw

In [ ]:
path_raw = bucket_raw
df_raw = spark.read.parquet(path_raw)
df_raw.createOrReplaceTempView("raw_base_telco")

print(log(), "Registros na Raw:", df_raw.count())
df_raw.show(5, truncate=False)


2026-02-10 12:05:49 >>> Registros na Raw: 1367104
+-----------+------+---------------+---+----+---------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+
|NUM_CPF    |SAFRA |FLAG_INSTALACAO|FPD|PROD|flag_mig2|var_26|var_27|var_28|var_29|var_30|var_31|var_32|var_33|var_34|var_35|var_36|var_37|var_38|var_39|var_40|var_41|var_42|var_43|var_44|var_45|var_46|var_47|var_48|var_49|var_50|var_51|var_52|var_53|var_54|var_55|var_56|var_57|var_58|var_59|var_60|var_61|var_62|var_63|var_64|var_65|var_66|var_67|var_68|var_69|var_70|var_71|var_72|var_73|var_74|var_75|var_76|var_77

# Processamento tipagem para camada Trusted

In [ ]:
df_trusted = spark.sql(f"""
    SELECT
        '{dthproc}' AS ts_proc,
        '{dthproc}' AS ts_proc_partition,
        CAST(NUM_CPF AS STRING) AS NUM_CPF,
        CAST(SAFRA AS INT) AS SAFRA,
        CAST(SUBSTRING(CAST(SAFRA AS STRING), 1, 4) AS INT) AS SAFRA_ANO,
        CAST(SUBSTRING(CAST(SAFRA AS STRING), 5, 2) AS INT) AS SAFRA_MES,
        CAST(FLAG_INSTALACAO AS BOOLEAN) AS IsSetup,
        CAST(FPD AS BOOLEAN) AS IsFPD,
        CAST(PROD AS STRING) AS ProductDescription,
        CAST(flag_mig2 AS STRING) AS ProductMigration
        --{",".join([f"CAST(var_{i} AS FLOAT) AS Var{i}" for i in range(26,94)])}
    FROM raw_base_telco
""")

df_trusted.createOrReplaceTempView("lake_base_telco")
df_trusted.cache()

print(log(), "Registros Trusted:", df_trusted.count())
#df_trusted.printSchema()
df_trusted.show(5, truncate=False)

2026-02-10 12:07:38 >>> Registros Trusted: 1367104
+--------------+-----------------+-----------+------+---------+---------+-------+-----+------------------+----------------+
|ts_proc       |ts_proc_partition|NUM_CPF    |SAFRA |SAFRA_ANO|SAFRA_MES|IsSetup|IsFPD|ProductDescription|ProductMigration|
+--------------+-----------------+-----------+------+---------+---------+-------+-----+------------------+----------------+
|20260210090530|20260210090530   |7779ZY78YXT|202411|2024     |11       |true   |true |CMV               |PRE             |
|20260210090530|20260210090530   |777YXTNZWZN|202410|2024     |10       |true   |false|CMV               |PRE             |
|20260210090530|20260210090530   |777ZXUXZZXT|202503|2025     |3        |true   |false|CMV               |PRE             |
|20260210090530|20260210090530   |777ZXZTUU7Y|202412|2024     |12       |true   |false|CMV               |PRE             |
|20260210090530|20260210090530   |7788N7YZN87|202412|2024     |12       |true   |

# Salvar na camada Trusted

In [ ]:
path_trusted = os.path.join(bucket_trusted, output_trusted)
print("Trusted path:", path_trusted)

df_trusted.write \
    .partitionBy("SAFRA","ts_proc_partition") \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(path_trusted)

Trusted path: /content/gdrive/Othercomputers/Meu laptop/Database_trusted/base_telco/trusted_base_telco


# Controle de carga

In [ ]:
controle = spark.sql(f"""
    SELECT
        '{output_trusted}' AS name_file,
        ts_proc,
        ts_proc_partition,
        COUNT(*) AS qtd_registros
    FROM lake_base_telco
    GROUP BY 1,2,3
""")

controle.createOrReplaceTempView("controle")
controle.cache()

print(log(), "Registros controle:", controle.count())
controle.show(truncate=False)

2026-02-10 12:14:20 >>> Registros controle: 1
+------------------+--------------+-----------------+-------------+
|name_file         |ts_proc       |ts_proc_partition|qtd_registros|
+------------------+--------------+-----------------+-------------+
|trusted_base_telco|20260210090530|20260210090530   |1367104      |
+------------------+--------------+-----------------+-------------+



# Controle de processamento

In [ ]:
path_control = os.path.join(bucket_control, f"tb_0002_controle_processamento_{bucket_base}_trusted")
print("Control path:", path_control)

controle.write \
    .mode("append") \
    .option("compression", "snappy") \
    .parquet(path_control)

Control path: /content/gdrive/Othercomputers/Meu laptop/Database_control/base_telco/tb_0002_controle_processamento_base_telco_trusted
